# Polymarket Calibration Analysis

**Question:** Are Polymarket prediction markets well calibrated?

A perfectly calibrated market means: markets trading at 70% odds should resolve YES ~70% of the time.
We test this across categories and find systematic mispricings.

**Data:** Resolved YES/NO markets from Polymarket Gamma API (2023–2025), daily price history from CLOB API.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT))

from lib.backtest import brier_score, brier_skill_score, calibration_bins
from lib.data_utils import load_parquet

DATA_DIR = Path().resolve().parent / "data"
PLOTS_DIR = Path().resolve().parent / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

# Publication style
plt.rcParams.update({
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})
CATEGORY_PALETTE = sns.color_palette("tab10")

## 1. Load & Prepare Data

In [ ]:
markets = load_parquet(DATA_DIR / "markets.parquet")
prices  = load_parquet(DATA_DIR / "price_history.parquet")

print(f"Markets: {len(markets):,}  |  Price rows: {len(prices):,}")
print(f"Markets with price history: {prices['market_id'].nunique():,}")
markets.head(3)

In [ ]:
# Compute per-market summary stats from price history
def market_price_stats(prices: pd.DataFrame) -> pd.DataFrame:
    """For each market: final-day odds, average odds, time-weighted average."""
    out = []
    for mid, grp in prices.sort_values("timestamp").groupby("market_id"):
        p = grp["price"].values
        out.append({
            "market_id": mid,
            "final_odds": p[-1],
            "avg_odds":   p.mean(),
            "median_odds": np.median(p),
            "n_days":     len(p),
            "odds_std":   p.std(),
        })
    return pd.DataFrame(out)

price_stats = market_price_stats(prices)
print(f"Markets with price stats: {len(price_stats):,}")
price_stats.describe().round(3)

In [ ]:
# Join: keep only markets that have both metadata AND price history
df = markets.merge(price_stats, on="market_id", how="inner")
df["y"] = (df["outcome"] == "YES").astype(int)  # binary label

# Clean category labels
df["category"] = df["category"].fillna("Unknown").str.strip()

print(f"Analysis dataset: {len(df):,} markets")
print(f"YES: {df['y'].sum()} ({df['y'].mean():.1%})   NO: {(1-df['y']).sum()} ({(1-df['y']).mean():.1%})")
print()
print(df["category"].value_counts().to_string())

## 2. Overall Calibration

We bin markets by their **final-day implied probability** and compare against the actual resolution rate.
A perfectly calibrated market lies on the diagonal.

In [ ]:
calib = calibration_bins(df["y"].values, df["final_odds"].values, n_bins=10)
calib

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

# Perfect calibration diagonal
ax.plot([0, 1], [0, 1], "--", color="grey", lw=1.5, label="Perfect calibration", zorder=1)

# Actual calibration
valid = calib.dropna(subset=["mean_forecast", "actual_rate"])
sc = ax.scatter(
    valid["mean_forecast"], valid["actual_rate"],
    s=valid["count"] * 6,      # bubble size = number of markets in bin
    alpha=0.85, zorder=3,
    c=valid["count"], cmap="viridis", edgecolors="white", linewidths=0.8,
)
ax.plot(valid["mean_forecast"], valid["actual_rate"], "-o",
        color="steelblue", lw=1.5, ms=0, zorder=2, alpha=0.6)

plt.colorbar(sc, ax=ax, label="Markets in bin", shrink=0.8)

bs  = brier_score(df["y"].values, df["final_odds"].values)
bss = brier_skill_score(df["y"].values, df["final_odds"].values)

ax.set_xlabel("Implied probability (final-day odds)")
ax.set_ylabel("Actual resolution rate (YES)")
ax.set_title(f"Polymarket Calibration\nBrier Score = {bs:.4f}  |  Skill Score = {bss:.3f}")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.legend(loc="upper left", framealpha=0.9)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

fig.tight_layout()
fig.savefig(PLOTS_DIR / "01_calibration_overall.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Brier Score: {bs:.4f}  (lower = better, naive baseline = {brier_score(df['y'].values, np.full(len(df), df['y'].mean())):.4f})")
print(f"Brier Skill: {bss:.3f}  (1=perfect, 0=no skill vs base rate)")

## 3. Calibration by Category

Are some categories better calibrated than others? 
Political markets are often cited as harder to calibrate due to tail events and polling noise.

In [ ]:
MIN_MARKETS_PER_CATEGORY = 5
cat_counts = df["category"].value_counts()
valid_cats = cat_counts[cat_counts >= MIN_MARKETS_PER_CATEGORY].index.tolist()

cat_stats = []
for cat in valid_cats:
    sub = df[df["category"] == cat]
    bs  = brier_score(sub["y"].values, sub["final_odds"].values)
    bss = brier_skill_score(sub["y"].values, sub["final_odds"].values)
    cat_stats.append({
        "category": cat,
        "n_markets": len(sub),
        "yes_rate": sub["y"].mean(),
        "brier_score": bs,
        "brier_skill": bss,
        "avg_final_odds": sub["final_odds"].mean(),
    })

cat_df = pd.DataFrame(cat_stats).sort_values("brier_score")
cat_df.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Brier score by category (lower = better)
ax = axes[0]
colors = [CATEGORY_PALETTE[i % 10] for i in range(len(cat_df))]
bars = ax.barh(cat_df["category"], cat_df["brier_score"], color=colors, alpha=0.85)
ax.axvline(bs, color="black", linestyle="--", lw=1.2, label=f"Overall ({bs:.4f})")
ax.set_xlabel("Brier Score (lower = better)")
ax.set_title("Brier Score by Category")
ax.legend()
for bar, val in zip(bars, cat_df["brier_score"]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=9)

# Right: calibration curves by category
ax = axes[1]
ax.plot([0, 1], [0, 1], "--", color="grey", lw=1.2, zorder=1, label="Perfect")
for i, cat in enumerate(valid_cats):
    sub = df[df["category"] == cat]
    if len(sub) < MIN_MARKETS_PER_CATEGORY:
        continue
    cb = calibration_bins(sub["y"].values, sub["final_odds"].values, n_bins=5)
    cb = cb.dropna(subset=["mean_forecast", "actual_rate"])
    if len(cb) < 2:
        continue
    color = CATEGORY_PALETTE[i % 10]
    ax.plot(cb["mean_forecast"], cb["actual_rate"],
            "-o", color=color, ms=5, lw=1.5,
            label=f"{cat} (n={len(sub)})", alpha=0.85)

ax.set_xlabel("Implied probability")
ax.set_ylabel("Actual resolution rate")
ax.set_title("Calibration Curves by Category")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.legend(fontsize=9, loc="upper left")

fig.tight_layout()
fig.savefig(PLOTS_DIR / "02_calibration_by_category.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Biggest Mispricings

Which markets had the largest divergence between implied probability and actual outcome?

- **Overpriced YES**: market said >50% but resolved NO
- **Underpriced YES**: market said <50% but resolved YES

In [ ]:
df["error"]     = df["final_odds"] - df["y"]          # positive = overconfident in YES
df["abs_error"] = df["error"].abs()

top_n = 10
cols  = ["question", "category", "final_odds", "outcome", "error", "volume_usd"]

print("=== TOP 10 BIGGEST MISPRICINGS (by |final_odds - outcome|) ===")
top_mispriced = df.nlargest(top_n, "abs_error")[cols].copy()
top_mispriced["final_odds"] = top_mispriced["final_odds"].map("{:.1%}".format)
top_mispriced["error"]      = top_mispriced["error"].map("{:+.2f}".format)
top_mispriced["volume_usd"] = top_mispriced["volume_usd"].map("${:,.0f}".format)
top_mispriced["question"]   = top_mispriced["question"].str[:70]
print(top_mispriced.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: error distribution
ax = axes[0]
ax.hist(df["error"], bins=20, color="steelblue", alpha=0.8, edgecolor="white")
ax.axvline(0, color="black", lw=1.5, linestyle="--", label="Perfect")
ax.axvline(df["error"].mean(), color="tomato", lw=1.5,
           label=f"Mean error = {df['error'].mean():+.3f}")
ax.set_xlabel("Final odds − Outcome  (positive = overpriced YES)")
ax.set_ylabel("Number of markets")
ax.set_title("Distribution of Mispricing")
ax.legend()

# Right: top-N worst mispricings as horizontal bar chart
ax = axes[1]
worst = df.nlargest(top_n, "abs_error").copy()
worst["label"] = worst["question"].str[:45] + "..."
worst["label"] = worst["label"] + "  (" + worst["outcome"] + ")"
colors = ["tomato" if e > 0 else "steelblue" for e in worst["error"]]
ax.barh(range(len(worst)), worst["error"], color=colors, alpha=0.85)
ax.set_yticks(range(len(worst)))
ax.set_yticklabels(worst["label"], fontsize=8)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Final odds − Outcome")
ax.set_title("10 Biggest Mispricings")
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="tomato",    label="Overpriced YES"),
    Patch(color="steelblue", label="Underpriced YES"),
], fontsize=9)

fig.tight_layout()
fig.savefig(PLOTS_DIR / "03_mispricings.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Price Evolution — How Fast Do Markets Converge?

Do Polymarket odds converge to the correct probability quickly, or do they drift?
We normalize each market's lifetime to [0, 1] and plot price paths by resolution outcome.

In [ ]:
# Normalize each market's price path to fraction of lifetime elapsed
norm_paths = []
for mid, grp in prices.sort_values("timestamp").groupby("market_id"):
    meta = df[df["market_id"] == mid]
    if meta.empty:
        continue
    outcome = meta["y"].iloc[0]
    p = grp["price"].values
    if len(p) < 3:
        continue
    t_norm = np.linspace(0, 1, len(p))
    norm_paths.append({"market_id": mid, "outcome": outcome,
                       "t": t_norm, "p": p})

print(f"Markets with normalized paths: {len(norm_paths)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, label, color, title in [
    (axes[0], 1, "steelblue",  "Markets resolving YES"),
    (axes[1], 0, "tomato",     "Markets resolving NO"),
]:
    subset = [p for p in norm_paths if p["outcome"] == label]
    
    # Individual paths (light)
    for path in subset:
        ax.plot(path["t"], path["p"], color=color, alpha=0.12, lw=0.8)
    
    # Compute mean path via interpolation onto common grid
    if subset:
        t_grid = np.linspace(0, 1, 50)
        interp_matrix = np.array([
            np.interp(t_grid, p["t"], p["p"]) for p in subset
        ])
        mean_path   = interp_matrix.mean(axis=0)
        median_path = np.median(interp_matrix, axis=0)
        q25 = np.percentile(interp_matrix, 25, axis=0)
        q75 = np.percentile(interp_matrix, 75, axis=0)
        
        ax.fill_between(t_grid, q25, q75, color=color, alpha=0.15, label="IQR")
        ax.plot(t_grid, mean_path,   color=color, lw=2.5, label="Mean",   zorder=3)
        ax.plot(t_grid, median_path, color=color, lw=1.5, linestyle="--", label="Median", zorder=3)
    
    ax.axhline(0.5, color="grey", lw=1, linestyle=":")
    ax.set_xlabel("Fraction of market lifetime elapsed")
    ax.set_ylabel("YES price (implied probability)")
    ax.set_title(f"{title} (n={len(subset)})")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.02, 1.02)
    ax.legend(fontsize=9)

fig.suptitle("Price Paths Normalized by Market Lifetime", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "04_price_convergence.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Summary & Key Findings

In [ ]:
best_cat  = cat_df.iloc[0]
worst_cat = cat_df.iloc[-1]
worst_mkt = df.nlargest(1, "abs_error").iloc[0]

print("=" * 60)
print("KEY FINDINGS")
print("=" * 60)
print(f"Dataset:   {len(df):,} resolved binary markets")
print(f"           {prices['market_id'].nunique():,} markets with price history")
print()
print(f"Overall Brier Score : {bs:.4f}")
print(f"Overall Skill Score : {bss:.3f}")
print(f"Mean error          : {df['error'].mean():+.4f}  "
      f"({'overestimates YES' if df['error'].mean()>0 else 'underestimates YES'})")
print()
print(f"Best calibrated  : {best_cat['category']}  (BS={best_cat['brier_score']:.4f}, n={int(best_cat['n_markets'])})")
print(f"Worst calibrated : {worst_cat['category']}  (BS={worst_cat['brier_score']:.4f}, n={int(worst_cat['n_markets'])})")
print()
print(f"Biggest mispricing: '{worst_mkt['question'][:70]}'")
print(f"  Final odds={worst_mkt['final_odds']:.1%}, Outcome={worst_mkt['outcome']}, Error={worst_mkt['error']:+.2f}")